# 📊 PHẦN 1: KHẢO SÁT DỮ LIỆU (EDA) & TIỀN XỬ LÝ (PREPROCESSING)
### Đề tài: Hệ thống phát hiện xâm nhập mạng lai (Hybrid ML-NIDS)
---
### Mục tiêu & Phép biến đổi toán học cốt lõi:
1. **Khám phá phân phối NSL-KDD (`125.973` dòng):** Nhận diện mất cân bằng lớp (`Normal`, `DoS`, `Probe`, `R2L`, `U2R`).
2. **Biến đổi Log1p:** $y = \ln(1 + \max(0, x))$ để nén đuôi phân phối lệch phải (Right-Skewness).
3. **Mã hóa One-Hot Encoding:** Chuyển đổi thuộc tính định danh sang vector nhị phân trực giao $\mathbf{v} \in \{0, 1\}^C$.
4. **Chuẩn hóa Z-Score (`StandardScaler`):** $z = \frac{x - \mu}{\sigma}$ đưa các đặc trưng số về trung bình $0$, phương sai $1$.


In [21]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Định nghĩa đường dẫn dữ liệu
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
TRAIN_FILE = DATA_DIR / "KDDTrain+.txt"
if not TRAIN_FILE.exists():
    TRAIN_FILE = Path(r"D:\data_miner\archive\KDDTrain+.txt")

print(f"✅ Đường dẫn dữ liệu huấn luyện: {TRAIN_FILE}")


✅ Đường dẫn dữ liệu huấn luyện: D:\data_miner\dtd\data\KDDTrain+.txt


### 1. Nạp dữ liệu và phân nhóm 41 đặc trưng gói tin
- **43 cột dữ liệu thô:** $41$ đặc trưng mạng + `label` (nhãn chi tiết) + `difficulty` (độ khó). Các biến `label` được ánh xạ về **5 nhóm hành vi chuẩn**: `Normal`, `DoS`, `Probe`, `R2L`, `U2R`.
- **Phân tách biến đầu vào:** $38$ biến số (`NUMERIC_COLS`, trong đó $3$ biến lệch phải nặng là `SKEWED_COLS`) và $3$ biến chữ danh định (`CATEGORICAL_COLS`).


In [22]:
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']
SKEWED_COLS = ['duration', 'src_bytes', 'dst_bytes']
NUMERIC_COLS = [c for c in COLUMN_NAMES if c not in CATEGORICAL_COLS + ['label', 'difficulty']]

ATTACK_MAP = {
    'normal': 'Normal',
    'neptune': 'DoS', 'smurf': 'DoS', 'pod': 'DoS', 'teardrop': 'DoS', 'land': 'DoS', 'back': 'DoS',
    'apache2': 'DoS', 'udpstorm': 'DoS', 'processtable': 'DoS', 'mailbomb': 'DoS',
    'portsweep': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe', 'satan': 'Probe', 'saint': 'Probe', 'mscan': 'Probe',
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'phf': 'R2L', 'multihop': 'R2L',
    'warezmaster': 'R2L', 'warezclient': 'R2L', 'spy': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L',
    'snmpguess': 'R2L', 'snmpgetattack': 'R2L', 'httptunnel': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'worm': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R', 'ps': 'U2R', 'xterm': 'U2R', 'sqlattack': 'U2R'
}

# Nạp toàn bộ 125,973 dòng của tập KDDTrain+
df = pd.read_csv(TRAIN_FILE, names=COLUMN_NAMES, header=None)
for c in CATEGORICAL_COLS:
    df[c] = df[c].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.strip().str.lower()
df['attack_cat'] = df['label'].map(lambda x: ATTACK_MAP.get(x, 'Normal'))

print(f"Tổng số mẫu nạp: {len(df):,} | Kích thước đầy đủ: {df.shape}")
df.head(50)


### 2. Trực quan hóa phân bố lưu lượng và hiệu quả của phép biến đổi Log1p
- **Hàm nén Log1p:** $y = \ln(1 + \max(0, x))$
- **Tác dụng toán học:** Đạo hàm $\frac{dy}{dx} = \frac{1}{1 + x}$ giảm rất nhanh khi $x$ lớn, giúp nén dải giá trị cực rộng của `src_bytes` từ $[0, 10^7]$ về $[0, 16.1]$, làm phân phối gần dạng chuẩn Gauss.


In [23]:
# In bảng đối chiếu phân bố Tập Train vs Tập Test (khớp Slide 6)
TEST_FILE = DATA_DIR / "KDDTest+.txt"
if not TEST_FILE.exists():
    TEST_FILE = Path(r"D:\data_miner\archive\KDDTest+.txt")
df_test_eda = pd.read_csv(TEST_FILE, names=COLUMN_NAMES, header=None)
df_test_eda['attack_cat'] = df_test_eda['label'].astype(str).str.strip().str.lower().map(lambda x: ATTACK_MAP.get(x, 'Normal'))

summary_tbl = pd.DataFrame({
    'Tập Train (KDDTrain+)': df['attack_cat'].value_counts(),
    'Tập Test (KDDTest+)': df_test_eda['attack_cat'].value_counts()
}).loc[['Normal', 'DoS', 'Probe', 'R2L', 'U2R']]
print("=== BẢNG PHÂN BỐ CÁC LỚP: TẬP TRAIN VS TẬP TEST ===")
display(summary_tbl)

fig, ax = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

# Biểu đồ 1: Phân bố các nhóm tấn công
counts = df['attack_cat'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']
bars = ax[0].bar(counts.index, counts.values, color=colors[:len(counts)], edgecolor='black', alpha=0.85)
ax[0].set_title("Phân Bố Các Lớp Lưu Lượng Mạng (NSL-KDD Train)", fontsize=12, fontweight='bold')
ax[0].set_ylabel("Số lượng mẫu")
ax[0].set_ylim(0, counts.max() * 1.12)
for bar in bars:
    val = int(bar.get_height())
    ax[0].text(bar.get_x() + bar.get_width() / 2, val + 800, f"{val:,}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# Biểu đồ 2: So sánh trước và sau biến đổi Log1p
raw_bytes = df['src_bytes'].clip(upper=5000)
log_bytes = np.log1p(df['src_bytes'])
sns.kdeplot(raw_bytes, ax=ax[1], color='crimson', fill=True, label='Gốc (Lệch phải cực đoan)', alpha=0.4)
sns.kdeplot(log_bytes * 500, ax=ax[1], color='teal', fill=True, label='Sau Log1p (Gần chuẩn Gauss)', alpha=0.4)
ax[1].set_title("Hiệu Quả Biến Đổi Log1p Trên `src_bytes`", fontsize=12, fontweight='bold')
ax[1].legend()

plt.tight_layout()
plt.show()


### 3. Pipeline tiền xử lý hoàn chỉnh (Log1p + One-Hot + StandardScaler)
- **Chuẩn hóa Z-Score (`StandardScaler`):** Với mỗi cột số thứ $j$:
  $$\mu_j = \frac{1}{N}\sum_{i=1}^{N} x_{ij}, \qquad \sigma_j = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(x_{ij} - \mu_j)^2}, \qquad z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$
- **One-Hot Encoding:** Biến $3$ cột danh định (`protocol_type`, `service`, `flag`) thành $84$ cột nhị phân $\mathbf{v} \in \{0,1\}^C$, đảm bảo khoảng cách Euclid giữa $2$ giao thức bất kỳ luôn đồng đều ($d = \sqrt{2}$).
- **Ghép ma trận đầu ra:** $X = \big[ Z_{\text{num}} \;\big|\; V_{\text{cat}} \big] \in \mathbb{R}^{125973 \times 122}$.


In [24]:
class NetworkPreprocessor:
    def __init__(self):
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.scaler = StandardScaler()

    def fit_transform(self, df):
        df_num = df[NUMERIC_COLS].copy()
        for col in SKEWED_COLS:
            if col in df_num.columns:
                df_num[col] = np.log1p(np.maximum(0, df_num[col].astype(np.float32)))
                
        num_scaled = self.scaler.fit_transform(df_num.values).astype(np.float32)
        cat_encoded = self.encoder.fit_transform(df[CATEGORICAL_COLS]).astype(np.float32)
        return np.hstack([num_scaled, cat_encoded]).astype(np.float32)

prep = NetworkPreprocessor()
X = prep.fit_transform(df)
y = df['attack_cat'].values
print(f"✅ Tiền xử lý hoàn tất! Ma trận đặc trưng X có kích thước: {X.shape}")


✅ Tiền xử lý hoàn tất! Ma trận đặc trưng X có kích thước: (125973, 122)
